# 03 Retina Super-Resolution with SRGAN

망막 OCT 영상(grayscale, 1채널)을 대상으로 SRGAN을 훈련합니다.

### 주요 수정 사항
| 항목 | 원본 코드 | 수정 후 |
|------|-----------|----------|
| 입력 채널 | 3 (RGB) | 1 (Grayscale) |
| LR 크기 | 24×24 | 64×64 |
| HR 크기 | 96×96 | 256×256 |
| 업스케일 배율 | ×4 | ×4 |
| VGG 입력 | 3ch 직접 입력 | 1ch → 3ch 복제 후 입력 |
| 훈련 안정화 | 없음 | D 업데이트 빈도 조절, LR 스케줄링 |


## 0. Google Drive 마운트 (Colab 전용)

데이터를 Google Drive에 미리 업로드한 뒤 아래 셀을 실행하세요.

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료')

## 1. 패키지 설치 및 라이브러리 임포트

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow.keras.layers import (
    Add, BatchNormalization, Conv2D, Dense, Flatten,
    Input, LeakyReLU, PReLU, Lambda, UpSampling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG19
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow:', tf.__version__)
print('GPU:', gpus if gpus else '없음 — 런타임 유형을 GPU로 변경하세요')

## 2. 하이퍼파라미터 및 경로 설정

In [ ]:
# =============================================
# 데이터 경로 설정 (Google Drive 기준)
# Drive에 아래 구조로 업로드되어 있어야 합니다:
#
# 내 드라이브/
# └── OCT_SR/
#     ├── High_Res/
#     │   ├── group1/  ← HR 이미지들
#     │   ├── group2/
#     │   └── group3/
#     └── Low_Res/
#         ├── group1/  ← LR 이미지들
#         ├── group2/
#         └── group3/
# =============================================
BASE_DIR = Path('/content/drive/MyDrive/OCT_SR')

HR_DIRS = [BASE_DIR / 'High_Res' / f'group{i}' for i in range(1, 4)]
LR_DIRS = [BASE_DIR / 'Low_Res'  / f'group{i}' for i in range(1, 4)]

# 체크포인트·결과는 Drive에 저장 (런타임 종료 후에도 유지)
CKPT_DIR   = Path('/content/drive/MyDrive/OCT_SR/checkpoints')
RESULT_DIR = Path('/content/drive/MyDrive/OCT_SR/results')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHANNELS  = 1
BATCH_SIZE      = 8
PRETRAIN_EPOCHS = 20
GAN_EPOCHS      = 100
D_UPDATE_FREQ   = 1
G_LR_INIT       = 1e-4
D_LR_INIT       = 1e-4
LAMBDA_CONTENT     = 1.0
LAMBDA_ADVERSARIAL = 1e-3
LAMBDA_PIXEL       = 0.0

print('경로 설정 완료')
for d in HR_DIRS + LR_DIRS:
    mark = '✓' if d.exists() else '✗ 없음'
    print(f'  [{mark}] {d}')

## 3. 데이터 로딩 및 전처리

실제 HR/LR 쌍 데이터를 직접 로드합니다.  
같은 group 폴더 내에서 파일명이 동일한 HR-LR 쌍을 매칭합니다.  
업스케일 배율(SCALE)은 실제 이미지 크기로 자동 감지됩니다.

In [ ]:
# --------------------------------------------------
# HR / LR 쌍 경로 수집
# 같은 group 내 동일 파일명으로 매칭
# --------------------------------------------------
EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def collect_pairs(hr_dirs, lr_dirs):
    """HR-LR 경로 쌍 리스트 반환 [(hr_path, lr_path), ...]"""
    pairs = []
    for hr_dir, lr_dir in zip(hr_dirs, lr_dirs):
        if not hr_dir.exists():
            print(f'[경고] HR 폴더 없음: {hr_dir}')
            continue
        if not lr_dir.exists():
            print(f'[경고] LR 폴더 없음: {lr_dir}')
            continue
        hr_files = {p.name: p for p in hr_dir.iterdir() if p.suffix.lower() in EXTS}
        lr_files = {p.name: p for p in lr_dir.iterdir() if p.suffix.lower() in EXTS}
        matched = sorted(set(hr_files) & set(lr_files))
        if not matched:
            # 파일명이 다른 경우 정렬 순서로 매칭
            hr_sorted = sorted(hr_files.values())
            lr_sorted = sorted(lr_files.values())
            n = min(len(hr_sorted), len(lr_sorted))
            pairs += list(zip(hr_sorted[:n], lr_sorted[:n]))
            print(f'[{hr_dir.name}] 파일명 불일치 → 정렬 순서로 {n}쌍 매칭')
        else:
            pairs += [(hr_files[n], lr_files[n]) for n in matched]
            print(f'[{hr_dir.name}] {len(matched)}쌍 매칭 완료')
    return pairs

all_pairs = collect_pairs(HR_DIRS, LR_DIRS)
print(f'\n총 쌍 수: {len(all_pairs)}')
if all_pairs:
    print(f'예시: HR={all_pairs[0][0].name}  LR={all_pairs[0][1].name}')

In [ ]:
# --------------------------------------------------
# 실제 이미지 크기 자동 감지 → SCALE, HR_SIZE, LR_SIZE 결정
# --------------------------------------------------
def load_gray(path: Path) -> np.ndarray:
    """grayscale로 읽어 [0,1] float32 반환"""
    return np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0

# 첫 번째 쌍으로 크기 확인
_hr_sample = Image.open(all_pairs[0][0]).convert('L')
_lr_sample = Image.open(all_pairs[0][1]).convert('L')
_hr_w, _hr_h = _hr_sample.size
_lr_w, _lr_h = _lr_sample.size

SCALE   = round(_hr_w / _lr_w)          # 예: 512/128 → 4
HR_SIZE = (_hr_h // (SCALE * 8)) * (SCALE * 8)   # 모델 호환 크기로 내림
LR_SIZE = HR_SIZE // SCALE

print(f'실제 HR 크기: {_hr_w}×{_hr_h}')
print(f'실제 LR 크기: {_lr_w}×{_lr_h}')
print(f'감지된 업스케일 배율: ×{SCALE}')
print(f'훈련 패치 크기: HR={HR_SIZE}×{HR_SIZE}, LR={LR_SIZE}×{LR_SIZE}')

assert SCALE in (2, 4), f'SCALE={SCALE}은 지원되지 않습니다. ×2 또는 ×4만 지원합니다.'

In [ ]:
# --------------------------------------------------
# HR / LR 쌍 로드 및 패치 추출
# --------------------------------------------------
def center_crop(img: np.ndarray, size: int) -> np.ndarray:
    """중앙 크롭"""
    h, w = img.shape
    top  = (h - size) // 2
    left = (w - size) // 2
    return img[top:top+size, left:left+size]

def random_crop_pair(hr: np.ndarray, lr: np.ndarray, hr_size: int, lr_size: int):
    """HR/LR를 동일 위치에서 랜덤 크롭 (공간 정합 유지)"""
    hr_h, hr_w = hr.shape
    if hr_h < hr_size or hr_w < hr_size:
        # 이미지가 작으면 리사이즈
        hr = np.array(Image.fromarray((hr*255).astype(np.uint8)).resize(
            (max(hr_w, hr_size), max(hr_h, hr_size)), Image.BICUBIC), dtype=np.float32) / 255.0
        lr = np.array(Image.fromarray((lr*255).astype(np.uint8)).resize(
            (max(hr_w, hr_size)//SCALE, max(hr_h, hr_size)//SCALE), Image.BICUBIC), dtype=np.float32) / 255.0
        hr_h, hr_w = hr.shape

    top_hr  = np.random.randint(0, hr_h - hr_size + 1)
    left_hr = np.random.randint(0, hr_w - hr_size + 1)
    top_lr  = top_hr  // SCALE
    left_lr = left_hr // SCALE

    hr_crop = hr[top_hr:top_hr+hr_size,   left_hr:left_hr+hr_size]
    lr_crop = lr[top_lr:top_lr+lr_size,   left_lr:left_lr+lr_size]
    return hr_crop, lr_crop


def build_dataset(pairs, val_split=0.1):
    hrs, lrs = [], []
    for hr_path, lr_path in tqdm(pairs, desc='데이터 로딩'):
        hr = load_gray(hr_path)
        lr = load_gray(lr_path)
        # LR 크기가 정확히 HR//SCALE이 아닐 경우 맞춤
        lr = np.array(Image.fromarray((lr*255).astype(np.uint8)).resize(
            (hr.shape[1]//SCALE, hr.shape[0]//SCALE), Image.BICUBIC), dtype=np.float32) / 255.0
        hr_patch, lr_patch = random_crop_pair(hr, lr, HR_SIZE, LR_SIZE)
        hrs.append(hr_patch[..., np.newaxis])
        lrs.append(lr_patch[..., np.newaxis])

    hrs = np.array(hrs, dtype=np.float32)
    lrs = np.array(lrs, dtype=np.float32)
    n_val = max(1, int(len(hrs) * val_split))
    # 셔플 후 분할
    perm = np.random.permutation(len(hrs))
    hrs, lrs = hrs[perm], lrs[perm]
    return lrs[n_val:], hrs[n_val:], lrs[:n_val], hrs[:n_val]


lr_train, hr_train, lr_val, hr_val = build_dataset(all_pairs)
print(f'훈련: LR {lr_train.shape}, HR {hr_train.shape}')
print(f'검증: LR {lr_val.shape},   HR {hr_val.shape}')

## 3-1. 샘플 시각화 (실제 LR / HR 쌍 확인)

In [ ]:
# --------------------------------------------------
# 정규화 유틸리티
# --------------------------------------------------
def normalize_m11(x):
    """[0,1] → [-1,1]"""
    return x * 2.0 - 1.0

def denormalize_m11(x):
    """[-1,1] → [0,1]"""
    return (x + 1.0) / 2.0

def pixel_shuffle(scale):
    """Sub-pixel convolution (depth-to-space)"""
    def fn(x):
        return tf.nn.depth_to_space(x, scale)
    return fn

In [ ]:
# --------------------------------------------------
# 샘플 시각화 — 실제 LR / Bicubic 업샘플 / HR 비교
# --------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
idx = 0

lr_img = lr_train[idx, :, :, 0]
hr_img = hr_train[idx, :, :, 0]
bicubic = np.array(
    Image.fromarray((lr_img * 255).astype(np.uint8))
    .resize((HR_SIZE, HR_SIZE), Image.BICUBIC), dtype=np.float32
) / 255.0
p = psnr(hr_img, bicubic, data_range=1.0)
s = ssim(hr_img, bicubic, data_range=1.0)

axes[0].imshow(lr_img, cmap='gray'); axes[0].set_title(f'실제 LR ({LR_SIZE}×{LR_SIZE})')
axes[1].imshow(bicubic, cmap='gray'); axes[1].set_title(f'Bicubic 기준선\nPSNR:{p:.2f}dB  SSIM:{s:.4f}')
axes[2].imshow(hr_img, cmap='gray'); axes[2].set_title(f'실제 HR ({HR_SIZE}×{HR_SIZE})')
for ax in axes: ax.axis('off')

plt.suptitle('데이터 샘플 확인 (훈련 전 Bicubic 기준선)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'sample_data.png', dpi=150)
plt.show()

In [ ]:
# --------------------------------------------------
# Generator (SRResNet) — grayscale 버전
# --------------------------------------------------
def upsample_block(x_in, num_filters):
    """Sub-pixel convolution으로 ×2 업샘플"""
    x = Conv2D(num_filters, kernel_size=3, padding='same')(x_in)
    x = Lambda(pixel_shuffle(scale=2))(x)
    return PReLU(shared_axes=[1, 2])(x)


def res_block(x_in, num_filters, momentum=0.8):
    x = Conv2D(num_filters, kernel_size=3, padding='same')(x_in)
    x = BatchNormalization(momentum=momentum)(x)
    x = PReLU(shared_axes=[1, 2])(x)
    x = Conv2D(num_filters, kernel_size=3, padding='same')(x)
    x = BatchNormalization(momentum=momentum)(x)
    return Add()([x_in, x])


def build_generator(num_filters=64, num_res_blocks=16, scale=4, channels=1):
    """
    입력: (B, LR_H, LR_W, 1) — grayscale, [0,1]
    출력: (B, HR_H, HR_W, 1) — grayscale, [0,1]
    """
    assert scale in (2, 4), 'scale must be 2 or 4'
    n_upsample = int(np.log2(scale))  # 4 → 2단계

    x_in = Input(shape=(None, None, channels), name='lr_input')
    x = Lambda(normalize_m11)(x_in)

    # 초기 특징 추출
    x = Conv2D(num_filters, kernel_size=9, padding='same')(x)
    x = x_skip = PReLU(shared_axes=[1, 2])(x)

    # Residual blocks
    for _ in range(num_res_blocks):
        x = res_block(x, num_filters)

    # Post-residual conv
    x = Conv2D(num_filters, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x_skip, x])

    # 업샘플링 단계 (×2 × n_upsample)
    for _ in range(n_upsample):
        # pixel shuffle: 채널 수가 scale^2 배여야 함 (각 단계 ×2이므로 4배)
        x = upsample_block(x, num_filters * 4)

    # 출력: grayscale 1채널
    x = Conv2D(channels, kernel_size=9, padding='same', activation='tanh')(x)
    x = Lambda(denormalize_m11)(x)

    return Model(x_in, x, name='Generator')


generator = build_generator(scale=SCALE, channels=CHANNELS)
generator.summary()

In [ ]:
# --------------------------------------------------
# Discriminator — grayscale 버전
# --------------------------------------------------
def discriminator_block(x_in, num_filters, strides=1, batchnorm=True, momentum=0.8):
    x = Conv2D(num_filters, kernel_size=3, strides=strides, padding='same')(x_in)
    if batchnorm:
        x = BatchNormalization(momentum=momentum)(x)
    return LeakyReLU(alpha=0.2)(x)


def build_discriminator(hr_size=HR_SIZE, num_filters=64, channels=1):
    """
    입력: (B, HR_H, HR_W, 1) — grayscale, [0,1]
    출력: (B, 1) — real/fake 확률
    """
    x_in = Input(shape=(hr_size, hr_size, channels), name='hr_input')
    x = Lambda(normalize_m11)(x_in)

    x = discriminator_block(x, num_filters, batchnorm=False)
    x = discriminator_block(x, num_filters, strides=2)

    x = discriminator_block(x, num_filters * 2)
    x = discriminator_block(x, num_filters * 2, strides=2)

    x = discriminator_block(x, num_filters * 4)
    x = discriminator_block(x, num_filters * 4, strides=2)

    x = discriminator_block(x, num_filters * 8)
    x = discriminator_block(x, num_filters * 8, strides=2)

    x = Flatten()(x)
    x = Dense(1024)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(1, activation='sigmoid')(x)

    return Model(x_in, x, name='Discriminator')


discriminator = build_discriminator(hr_size=HR_SIZE, channels=CHANNELS)
discriminator.summary()

In [ ]:
# --------------------------------------------------
# VGG Perceptual Loss 네트워크
# - VGG19는 3채널 입력만 허용
# - grayscale을 3채널로 tile하여 특징 추출
# - VGG22: block2_conv2 (5번째 레이어)
# - VGG54: block5_conv4 (20번째 레이어)
# --------------------------------------------------
def build_vgg_loss_network(output_layer_idx=5):
    """
    입력: (B, H, W, 3) — 3채널 (grayscale 복제 후 입력)
    출력: 중간 특징 맵
    """
    vgg = VGG19(input_shape=(None, None, 3), include_top=False, weights='imagenet')
    vgg.trainable = False
    return Model(vgg.input, vgg.layers[output_layer_idx].output, name='VGG_Loss')


vgg_loss_net = build_vgg_loss_network(output_layer_idx=5)  # VGG22
print('VGG loss 네트워크 로드 완료:', vgg_loss_net.output_shape)

## 5. 손실 함수 정의

$$\mathcal{L}_{SR} = \underbrace{\mathcal{L}_{content}}_{\text{VGG perceptual}} + \underbrace{\lambda_{adv}\, \mathcal{L}_{adv}}_{\text{adversarial}}$$

- **Content loss**: VGG 특징 공간에서의 MSE — 픽셀 단위 MSE보다 perceptually 선명한 결과
- **Adversarial loss**: Generator가 Discriminator를 속이는 능력 — 실제 분포에 가까운 텍스처 생성

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()
mse = tf.keras.losses.MeanSquaredError()


def perceptual_loss(hr_imgs, sr_imgs):
    """
    VGG feature space MSE.
    grayscale (B,H,W,1) → 3채널 복제 → VGG 입력
    VGG는 ImageNet 통계로 정규화된 [0,255] 입력을 기대하지만,
    여기서는 상대적인 feature 비교가 목적이므로 단순 tile 적용.
    """
    hr_3ch = tf.tile(hr_imgs, [1, 1, 1, 3])  # (B,H,W,3)
    sr_3ch = tf.tile(sr_imgs, [1, 1, 1, 3])
    hr_feat = vgg_loss_net(hr_3ch * 255.0, training=False)
    sr_feat = vgg_loss_net(sr_3ch * 255.0, training=False)
    return mse(hr_feat, sr_feat)


def adversarial_loss_g(fake_logits):
    """Generator: D가 SR 이미지를 real로 분류하도록 유도"""
    return bce(tf.ones_like(fake_logits), fake_logits)


def discriminator_loss(real_logits, fake_logits):
    """Discriminator: real=1, fake=0"""
    loss_real = bce(tf.ones_like(real_logits),  real_logits)
    loss_fake = bce(tf.zeros_like(fake_logits), fake_logits)
    return loss_real + loss_fake


def pixel_loss(hr_imgs, sr_imgs):
    return mse(hr_imgs, sr_imgs)


print('손실 함수 정의 완료')

## 6. 옵티마이저 설정

**훈련 안정화 전략:**
- G와 D의 Learning Rate를 동일하게 시작 후 Exponential Decay 적용
- D 업데이트 빈도(`D_UPDATE_FREQ`)로 G/D 균형 조절 (기본값 1:1)
- D가 너무 강해지면 `D_UPDATE_FREQ`를 줄이거나 D의 LR을 낮춤

In [ ]:
# 사전학습 (MSE only)
pretrain_optimizer = Adam(learning_rate=G_LR_INIT)
generator.compile(optimizer=pretrain_optimizer, loss='mse')

# GAN 훈련용 옵티마이저 (LR 스케줄 적용)
steps_per_epoch = max(1, len(lr_train) // BATCH_SIZE)
g_lr_schedule = ExponentialDecay(
    G_LR_INIT, decay_steps=steps_per_epoch * 50, decay_rate=0.5, staircase=True
)
d_lr_schedule = ExponentialDecay(
    D_LR_INIT, decay_steps=steps_per_epoch * 50, decay_rate=0.5, staircase=True
)

g_optimizer = Adam(learning_rate=g_lr_schedule, beta_1=0.9)
d_optimizer = Adam(learning_rate=d_lr_schedule, beta_1=0.9)

print(f'steps_per_epoch: {steps_per_epoch}')
print('옵티마이저 설정 완료')

## 7. 사전학습 — Generator (MSE Loss만 사용)

GAN 훈련 전 Generator를 pixel-wise MSE로 먼저 훈련합니다.  
이 단계를 건너뛰면 GAN 초기 훈련이 불안정해질 수 있습니다.

In [ ]:
pretrain_history = generator.fit(
    lr_train, hr_train,
    epochs=PRETRAIN_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(lr_val, hr_val),
    verbose=1
)

# 사전학습 곡선
plt.figure(figsize=(8, 4))
plt.plot(pretrain_history.history['loss'], label='Train MSE')
plt.plot(pretrain_history.history['val_loss'], label='Val MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Generator Pre-training (MSE)')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'pretrain_loss.png', dpi=150)
plt.show()

# 체크포인트 저장
generator.save_weights(str(CKPT_DIR / 'generator_pretrained.weights.h5'))
print('사전학습 완료')

## 8. GAN 훈련 (Perceptual + Adversarial Loss)

In [ ]:
@tf.function
def train_step_d(lr_batch, hr_batch):
    """Discriminator 업데이트"""
    sr_batch = generator(lr_batch, training=False)
    with tf.GradientTape() as tape:
        real_out = discriminator(hr_batch, training=True)
        fake_out = discriminator(sr_batch, training=True)
        d_loss   = discriminator_loss(real_out, fake_out)
    grads = tape.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))
    return d_loss


@tf.function
def train_step_g(lr_batch, hr_batch):
    """Generator 업데이트 (perceptual + adversarial)"""
    with tf.GradientTape() as tape:
        sr_batch   = generator(lr_batch, training=True)
        fake_out   = discriminator(sr_batch, training=False)

        loss_adv   = adversarial_loss_g(fake_out)
        loss_perc  = perceptual_loss(hr_batch, sr_batch)
        loss_pix   = pixel_loss(hr_batch, sr_batch)

        g_loss = (LAMBDA_CONTENT * loss_perc
                  + LAMBDA_ADVERSARIAL * loss_adv
                  + LAMBDA_PIXEL * loss_pix)
    grads = tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
    return g_loss, loss_perc, loss_adv


print('훈련 스텝 함수 정의 완료')

In [ ]:
# --------------------------------------------------
# GAN 훈련 루프
# --------------------------------------------------
history = {'g_loss': [], 'd_loss': [], 'perc_loss': [], 'adv_loss': [],
           'val_psnr': [], 'val_ssim': []}

n_train = len(lr_train)
best_psnr = 0.0

for epoch in range(1, GAN_EPOCHS + 1):
    # 배치 인덱스 셔플
    idx_perm = np.random.permutation(n_train)

    epoch_g, epoch_d, epoch_perc, epoch_adv = [], [], [], []

    for step in range(steps_per_epoch):
        batch_idx = idx_perm[step * BATCH_SIZE: (step + 1) * BATCH_SIZE]
        lr_batch  = lr_train[batch_idx]
        hr_batch  = hr_train[batch_idx]

        # Discriminator 업데이트 (D_UPDATE_FREQ 회)
        for _ in range(D_UPDATE_FREQ):
            d_loss = train_step_d(lr_batch, hr_batch)

        # Generator 업데이트 (1회)
        g_loss, perc_loss, adv_loss = train_step_g(lr_batch, hr_batch)

        epoch_g.append(float(g_loss))
        epoch_d.append(float(d_loss))
        epoch_perc.append(float(perc_loss))
        epoch_adv.append(float(adv_loss))

    # Validation PSNR/SSIM
    sr_val = generator.predict(lr_val, batch_size=BATCH_SIZE, verbose=0)
    sr_val = np.clip(sr_val, 0.0, 1.0)
    val_psnrs = [psnr(hr_val[i,:,:,0], sr_val[i,:,:,0], data_range=1.0) for i in range(len(hr_val))]
    val_ssims = [ssim(hr_val[i,:,:,0], sr_val[i,:,:,0], data_range=1.0) for i in range(len(hr_val))]
    mean_psnr = np.mean(val_psnrs)
    mean_ssim = np.mean(val_ssims)

    history['g_loss'].append(np.mean(epoch_g))
    history['d_loss'].append(np.mean(epoch_d))
    history['perc_loss'].append(np.mean(epoch_perc))
    history['adv_loss'].append(np.mean(epoch_adv))
    history['val_psnr'].append(mean_psnr)
    history['val_ssim'].append(mean_ssim)

    print(f'Epoch {epoch:03d}/{GAN_EPOCHS} '
          f'G:{np.mean(epoch_g):.4f} D:{np.mean(epoch_d):.4f} '
          f'Perc:{np.mean(epoch_perc):.4f} Adv:{np.mean(epoch_adv):.4f} '
          f'PSNR:{mean_psnr:.2f}dB SSIM:{mean_ssim:.4f}')

    # Best 모델 저장
    if mean_psnr > best_psnr:
        best_psnr = mean_psnr
        generator.save_weights(str(CKPT_DIR / 'generator_best.weights.h5'))

    # 10 에폭마다 체크포인트
    if epoch % 10 == 0:
        generator.save_weights(str(CKPT_DIR / f'generator_epoch{epoch:03d}.weights.h5'))
        discriminator.save_weights(str(CKPT_DIR / f'discriminator_epoch{epoch:03d}.weights.h5'))

print(f'\n훈련 완료! Best PSNR: {best_psnr:.2f} dB')

## 9. 학습 곡선 시각화

In [ ]:
epochs_range = range(1, GAN_EPOCHS + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# G / D total loss
axes[0, 0].plot(epochs_range, history['g_loss'], label='Generator Loss', color='steelblue')
axes[0, 0].plot(epochs_range, history['d_loss'], label='Discriminator Loss', color='tomato')
axes[0, 0].set_title('Generator & Discriminator Loss')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True)

# G 손실 분해
axes[0, 1].plot(epochs_range, history['perc_loss'],  label='Perceptual (VGG)', color='darkorange')
axes[0, 1].plot(epochs_range, history['adv_loss'],   label='Adversarial',      color='purple')
axes[0, 1].set_title('Generator Loss Breakdown')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(True)

# PSNR
axes[1, 0].plot(epochs_range, history['val_psnr'], color='green')
axes[1, 0].set_title('Validation PSNR')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('PSNR (dB)')
axes[1, 0].grid(True)

# SSIM
axes[1, 1].plot(epochs_range, history['val_ssim'], color='navy')
axes[1, 1].set_title('Validation SSIM')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('SSIM')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(RESULT_DIR / 'training_curves.png', dpi=150)
plt.show()

## 10. 결과 시각화 및 정량 평가

In [ ]:
# Best 가중치 로드
generator.load_weights(str(CKPT_DIR / 'generator_best.weights.h5'))

# 검증 세트 전체 추론
sr_outputs = generator.predict(lr_val, batch_size=BATCH_SIZE, verbose=0)
sr_outputs = np.clip(sr_outputs, 0.0, 1.0)

# 정량 지표 계산
psnr_sr, ssim_sr, psnr_bc, ssim_bc = [], [], [], []
for i in range(len(hr_val)):
    hr_img = hr_val[i, :, :, 0]
    sr_img = sr_outputs[i, :, :, 0]
    # Bicubic 기준선
    bc_img = np.array(
        Image.fromarray((lr_val[i, :, :, 0] * 255).astype(np.uint8))
        .resize((HR_SIZE, HR_SIZE), Image.BICUBIC), dtype=np.float32
    ) / 255.0
    psnr_sr.append(psnr(hr_img, sr_img, data_range=1.0))
    ssim_sr.append(ssim(hr_img, sr_img, data_range=1.0))
    psnr_bc.append(psnr(hr_img, bc_img, data_range=1.0))
    ssim_bc.append(ssim(hr_img, bc_img, data_range=1.0))

print(f'[Bicubic] PSNR: {np.mean(psnr_bc):.2f} dB  SSIM: {np.mean(ssim_bc):.4f}')
print(f'[SRGAN]   PSNR: {np.mean(psnr_sr):.2f} dB  SSIM: {np.mean(ssim_sr):.4f}')

In [ ]:
# --------------------------------------------------
# 대표 결과 이미지 (LR / Bicubic / SR / HR)
# --------------------------------------------------
N_SHOW = min(4, len(hr_val))
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, N_SHOW * 4))
col_titles = ['LR Input', 'Bicubic Upscale', 'SRGAN Output', 'HR Ground Truth']

for i in range(N_SHOW):
    hr_img = hr_val[i, :, :, 0]
    lr_img = lr_val[i, :, :, 0]
    sr_img = sr_outputs[i, :, :, 0]
    bc_img = np.array(
        Image.fromarray((lr_img * 255).astype(np.uint8))
        .resize((HR_SIZE, HR_SIZE), Image.BICUBIC), dtype=np.float32
    ) / 255.0

    p_bc = psnr(hr_img, bc_img, data_range=1.0)
    s_bc = ssim(hr_img, bc_img, data_range=1.0)
    p_sr = psnr(hr_img, sr_img, data_range=1.0)
    s_sr = ssim(hr_img, sr_img, data_range=1.0)

    imgs    = [lr_img, bc_img, sr_img, hr_img]
    metrics = ['', f'PSNR:{p_bc:.2f}dB\nSSIM:{s_bc:.4f}',
               f'PSNR:{p_sr:.2f}dB\nSSIM:{s_sr:.4f}', '']

    for j, (img, metric) in enumerate(zip(imgs, metrics)):
        ax = axes[i, j] if N_SHOW > 1 else axes[j]
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if i == 0:
            ax.set_title(col_titles[j], fontsize=12, fontweight='bold')
        if metric:
            ax.set_xlabel(metric, fontsize=9)

plt.suptitle('SRGAN Retina OCT Super-Resolution Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'sr_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. 훈련 안정성 분석

GAN 훈련에서 G/D 균형이 깨지는 경우 아래 셀을 참고하여 하이퍼파라미터를 조정하세요.

In [ ]:
# G/D loss 비율 분석
g_arr = np.array(history['g_loss'])
d_arr = np.array(history['d_loss'])
ratio = g_arr / (d_arr + 1e-8)

plt.figure(figsize=(10, 4))
plt.plot(ratio, color='darkcyan', label='G_loss / D_loss')
plt.axhline(y=1.0, color='red', linestyle='--', label='균형점 (ratio=1)')
plt.xlabel('Epoch'); plt.ylabel('Loss Ratio (G/D)')
plt.title('GAN Training Balance (G/D Loss Ratio)')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'training_balance.png', dpi=150)
plt.show()

print('\n[권장 조정 가이드]')
print('  ratio >> 1 → G가 D보다 훨씬 강함 → D_LR 올리거나 D_UPDATE_FREQ 증가')
print('  ratio << 1 → D가 G보다 훨씬 강함 → D_LR 낮추거나 D_UPDATE_FREQ 감소')
print(f'  현재 평균 ratio: {ratio.mean():.3f}')

In [ ]:
# --------------------------------------------------
# 최종 정량 지표 요약
# --------------------------------------------------
print('=' * 55)
print(f'  방법           PSNR (dB)    SSIM')
print('=' * 55)
print(f'  Bicubic        {np.mean(psnr_bc):>8.2f}    {np.mean(ssim_bc):.4f}')
print(f'  SRGAN (Ours)   {np.mean(psnr_sr):>8.2f}    {np.mean(ssim_sr):.4f}')
print('=' * 55)
print(f'  PSNR 향상:  +{np.mean(psnr_sr) - np.mean(psnr_bc):.2f} dB')
print(f'  SSIM 향상:  +{np.mean(ssim_sr) - np.mean(ssim_bc):.4f}')